# CDS529 Phase 2 — Multi-Roleplayer × Multi-Judge Experiment

**3 Roleplayers** (CharacterGLM, ChatHaruhi, RoleLLaMA) × **4 Judges** (Claude, GPT-4o, Gemini, DeepSeek)

## How to use this notebook
1. `Runtime → Change runtime type → A100 GPU` (only needed for RoleLLaMA)
2. Upload `prompt_template_customized_new.docx` and `characterglm_roles.csv` to Colab (left sidebar → 📁)
3. Fill in your API keys in **Cell 2**
4. Run all cells in order
5. Uncomment the experiment combos you want in **Cell 6**

**No separate .py file needed** — the entire pipeline is embedded in this notebook.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q zhipuai anthropic openai google-generativeai python-docx python-dotenv chatharuhi nest_asyncio
# Uncomment below ONLY for RoleLLaMA experiments:
!pip install -q vllm
!pip install -q google-genai
!pip install langchain==0.1.20 langchain-openai==0.1.7 langchain-core==0.1.53 langchain-community==0.0.38


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.27.0 requires pyjwt[crypto]>=2.10.1, but you have pyjwt 2.8.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
zhipuai 2.1.5.20250825 requires pyjwt<2.9.0,>=2.8.0, but you have pyjwt 2.12.1 which is incompatible.
langchain 0

In [ ]:
# Cell 1 — ALL installs (run once, then restart runtime)
!pip install -q zhipuai anthropic python-docx python-dotenv chatharuhi nest_asyncio google-generativeai

# Pin openai to 1.x that works with BOTH vllm's OpenAI-compatible client AND langchain
!pip install -q "openai>=1.40.0,<2.0.0"

# vLLM — pin to a version that tolerates openai 1.x
!pip install -q "vllm<0.19.0"

# Langchain — these old pins are the reason you can't use openai 2.x
!pip install -q langchain==0.1.20 langchain-openai==0.1.7 langchain-core==0.1.53 langchain-community==0.0.38

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.27.0 requires pyjwt[crypto]>=2.10.1, but you have pyjwt 2.8.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
langchain-core 0.1.53 requires packaging<24.0,>=23.2, but you have packaging 26.1 which is incompatible.
zhipuai 

## Cell 2 — API Keys

⚠️ **Paste your real keys below.** Clear them before sharing this notebook.

Only fill in the ones you need:
- `ZHIPUAI_API_KEY` → `--roleplayer charglm`
- `OPENAI_API_KEY` → `--judge openai` or `--roleplayer chatharuhi`
- `ANTHROPIC_API_KEY` → `--judge claude`
- `GOOGLE_API_KEY` → `--judge gemini`
- `DEEPSEEK_API_KEY` → `--judge deepseek`

In [ ]:
import os

# ═══════════════════════════════════════════════
# PASTE YOUR API KEYS HERE
# ═══════════════════════════════════════════════
os.environ["NO_PROXY"] = "localhost,127.0.0.1,*.googleapis.com,generativelanguage.googleapis.com"
os.environ["ZHIPUAI_API_KEY"]   = ""   # CharacterGLM  — https://open.bigmodel.cn
os.environ["OPENAI_API_KEY"]    = ""   # GPT-4o        — https://platform.openai.com
os.environ["ANTHROPIC_API_KEY"] = ""   # Claude        — https://console.anthropic.com
os.environ["GOOGLE_API_KEY"]    = ""   # Gemini        — https://aistudio.google.com
os.environ["DEEPSEEK_API_KEY"]  = ""   # DeepSeek      — https://platform.deepseek.com

# ChatHaruhi backend LLM (default: openai = GPT-4o as backbone)
os.environ["CHATHARUHI_BACKEND"] = "openai"

# RoleLLaMA vLLM endpoint (auto-set in Cell 5, no need to change)
os.environ["ROLELLM_API_BASE"] = "http://localhost:8000/v1"

# ═══════════════════════════════════════════════
# Quick check
# ═══════════════════════════════════════════════
for key in ["ZHIPUAI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "DEEPSEEK_API_KEY"]:
    val = os.environ.get(key, "")
    status = f"✅ SET ({len(val)} chars)" if val else "❌ EMPTY"
    print(f"  {key}: {status}")


  ZHIPUAI_API_KEY: ✅ SET (49 chars)
  OPENAI_API_KEY: ✅ SET (164 chars)
  ANTHROPIC_API_KEY: ✅ SET (108 chars)
  GOOGLE_API_KEY: ✅ SET (39 chars)
  DEEPSEEK_API_KEY: ✅ SET (35 chars)


## Cell 3 — Verify Uploaded Files

In [ ]:
import os
os.chdir("/content")

for f in ["prompt_template_customized_new.docx", "characterglm_roles.csv"]:
    ok = os.path.exists(f)
    print(f"  {'✅' if ok else '❌'} {f}")
    if not ok:
        print(f"     → Upload via left sidebar 📁")

print(f"\nWorking dir: {os.getcwd()}")
print(f"Files: {[f for f in os.listdir('.') if not f.startswith('.')]}")


  ✅ prompt_template_customized_new.docx
  ✅ characterglm_roles.csv

Working dir: /content
Files: ['vllm_stdout.log', 'characterglm_roles.csv', 'results_all.zip', '=2.0', 'vllm_stderr.log', 'prompt_template_customized_new.docx', 'results', 'results.zip', 'sample_data']


## Cell 4 — Experiment Pipeline (full script)

This cell defines **all** functions: CSV loader, docx parser, API wrappers,
conversation runner, evaluator, and trial orchestrator.

**Just run it once** — it defines everything. The actual experiments are triggered in Cell 6.

In [ ]:

from __future__ import annotations
import asyncio
import csv
import json
import logging
import os
import re
import sys
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any
import langchain.chat_models
from langchain_openai import ChatOpenAI
langchain.chat_models.ChatOpenAI = ChatOpenAI
from google import genai

# ===========================================================================
# MODEL CONFIGURATION
# ===========================================================================

# ===========================================================================
# ===========================================================================

# Maps --roleplayer CLI value to config.
ROLEPLAYER_MODELS = {
    "charglm":    {"provider": "zhipuai",    "model": "charglm-4"},
    "chatharuhi": {"provider": "chatharuhi", "model": "chatharuhi"},     # backend LLM configurable
    "rolellm":    {"provider": "rolellm",    "model": "zephyr7788/RoleLLM"},   # served via vLLM
}

# Maps --judge CLI value to (provider, model_id).
JUDGE_MODELS = {
    "claude":   {"provider": "anthropic", "model": "claude-sonnet-4-5"},
    "openai":   {"provider": "openai",    "model": "gpt-4o"},
    "gemini":   {"provider": "gemini",    "model": "gemini-2.5-flash"},
    "deepseek": {"provider": "deepseek",  "model": "deepseek-chat"},
}

ROLEPLAYER_MAX_TOKENS = 400
ANTAGONIST_MAX_TOKENS = 800
EVALUATOR_MAX_TOKENS = 8000

# ===========================================================================
# GROUND TRUTH MAPPING
# ===========================================================================
# PDB's four voting buckets mapped to 1-5 Likert. No "1" (very low) since PDB's
# lowest bucket is 25%, not 0%. Note as a limitation in the writeup.
PDB_TO_LIKERT = {25: 2, 50: 3, 75: 4, 100: 5}

TRAIT_NAMES = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("phase1")


# ===========================================================================
# DATA LOADING
# ===========================================================================

@dataclass
class GroundTruth:
    """PDB-derived canonical Big Five profile for one character."""
    character: str
    source_material: str
    openness: int
    conscientiousness: int
    extraversion: int
    agreeableness: int
    neuroticism: int
    confidence: dict[str, float]
    vote_counts: dict[str, int]

    def as_dict(self) -> dict[str, int]:
        return {
            "Openness": self.openness,
            "Conscientiousness": self.conscientiousness,
            "Extraversion": self.extraversion,
            "Agreeableness": self.agreeableness,
            "Neuroticism": self.neuroticism,
        }


def parse_pdb_label(label: str) -> int:
    """Convert 'Extraversion 75%' -> Likert int."""
    match = re.search(r"(\d+)\s*%", label)
    if not match:
        raise ValueError(f"Could not parse PDB label: {label!r}")
    pct = int(match.group(1))
    if pct not in PDB_TO_LIKERT:
        raise ValueError(f"Unexpected PDB percentage {pct} in label {label!r}")
    return PDB_TO_LIKERT[pct]


def load_ground_truth(csv_path: Path) -> dict[str, GroundTruth]:
    """Parse the CSV into GroundTruth objects keyed by character name.

    Supports TWO CSV schemas:
      A) Original: columns = name, work, most_voted_{Trait}, ratio_most_vote_count_{Trait}, total_vote_count_{Trait}
      B) New PDB:  columns = name, subcategory, big5_{trait} (e.g. "Openness 75%"), vote_count, ...
    Auto-detects which format is present based on column names.
    """
    truths: dict[str, GroundTruth] = {}

    # Try multiple encodings
    for enc in ("utf-8-sig", "utf-8", "Windows-1252", "latin-1"):
        try:
            with open(csv_path, encoding=enc) as f:
                raw_text = f.read()
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
    else:
        raise RuntimeError(f"Could not decode CSV file: {csv_path}")

    reader = csv.DictReader(raw_text.splitlines())
    cols = set(reader.fieldnames or [])

    # Detect schema
    is_original = "work" in cols and "most_voted_Openness" in cols
    is_new_pdb = "big5_openness" in cols

    if not is_original and not is_new_pdb:
        raise ValueError(
            f"CSV {csv_path} has unrecognized schema. "
            f"Expected either 'work'+'most_voted_Openness' columns or 'big5_openness' column. "
            f"Found columns: {sorted(cols)}"
        )

    # Column name mapping for new PDB format: big5_{lowercase} -> Trait Name
    BIG5_COL_MAP = {
        "Openness": "big5_openness",
        "Conscientiousness": "big5_conscientiousness",
        "Extraversion": "big5_extraversion",
        "Agreeableness": "big5_agreeableness",
        "Neuroticism": "big5_neuroticism",
    }

    for row in reader:
        name = row["name"].strip()

        if is_original:
            source = row["work"].strip()
            confidence: dict[str, float] = {}
            votes: dict[str, int] = {}
            scores: dict[str, int] = {}
            for trait in TRAIT_NAMES:
                scores[trait] = parse_pdb_label(row[f"most_voted_{trait}"])
                ratio_str = row[f"ratio_most_vote_count_{trait}"].rstrip("%")
                confidence[trait] = float(ratio_str) / 100.0
                votes[trait] = int(row[f"total_vote_count_{trait}"])
        else:
            # New PDB format
            source = row.get("subcategory", "").strip() or row.get("category", "").strip()
            confidence = {}
            votes = {}
            scores = {}
            total_votes = int(row.get("vote_count", 0) or 0)
            for trait in TRAIT_NAMES:
                col = BIG5_COL_MAP[trait]
                scores[trait] = parse_pdb_label(row[col])
                # New CSV has no per-trait confidence/votes; use placeholders
                confidence[trait] = 0.0  # unknown — flag in analysis
                votes[trait] = total_votes  # use global vote count as proxy

        truths[name] = GroundTruth(
            character=name,
            source_material=source,
            openness=scores["Openness"],
            conscientiousness=scores["Conscientiousness"],
            extraversion=scores["Extraversion"],
            agreeableness=scores["Agreeableness"],
            neuroticism=scores["Neuroticism"],
            confidence=confidence,
            vote_counts=votes,
        )
        log.info(
            "Loaded %s: O=%d C=%d E=%d A=%d N=%d (avg conf %.2f)",
            name, scores["Openness"], scores["Conscientiousness"],
            scores["Extraversion"], scores["Agreeableness"], scores["Neuroticism"],
            sum(confidence.values()) / len(confidence),
        )
    return truths


# ===========================================================================
# PROMPT EXTRACTION FROM DOCX
# ===========================================================================

@dataclass
class Cell:
    """One (character, scenario) cell's prompts."""
    character: str
    scenario_key: str
    scenario_title: str
    target_traits: str
    roleplayer_prompt: str
    antagonist_prompt: str


SCENARIO_TITLE_TO_KEY = {
    "The Defective Purchase": "defective_purchase",
    "The Apprentice's Confusion": "apprentice_confusion",
    "The Grave News": "grave_news",
    "The Authority Confrontation": "authority_confrontation",
    "The High-Stakes Evaluation": "high_stakes_evaluation",
    "The Peer Confrontation": "peer_confrontation",
}


def extract_cells_from_docx(docx_path: Path) -> dict[tuple[str, str], Cell]:
    """Parse the customized prompt docx into (character, scenario_key) -> Cell."""
    try:
        from docx import Document  # python-docx
    except ImportError:
        log.error("python-docx is required. Install with: pip install python-docx")
        sys.exit(1)

    doc = Document(str(docx_path))
    cells: dict[tuple[str, str], Cell] = {}

    current_character: str | None = None
    current_scenario_key: str | None = None
    current_scenario_title: str | None = None
    current_target_traits: str = ""
    current_subsection: str | None = None
    roleplayer_buf: list[str] = []
    antagonist_buf: list[str] = []

    def flush_cell() -> None:
        nonlocal roleplayer_buf, antagonist_buf
        if current_character and current_scenario_key:
            key = (current_character, current_scenario_key)
            cells[key] = Cell(
                character=current_character,
                scenario_key=current_scenario_key,
                scenario_title=current_scenario_title or "",
                target_traits=current_target_traits,
                roleplayer_prompt="\n\n".join(p for p in roleplayer_buf if p.strip()),
                antagonist_prompt="\n\n".join(p for p in antagonist_buf if p.strip()),
            )
        roleplayer_buf = []
        antagonist_buf = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        style = para.style.name if para.style else ""

        if style == "Heading 1" and text.startswith("Character:"):
            flush_cell()
            current_character = text.replace("Character:", "").strip()
            current_scenario_key = None
            current_subsection = None
            continue

        if style == "Heading 2" and text.startswith("Scenario"):
            flush_cell()
            match = re.match(r"Scenario\s+\d+:\s*(.+)", text)
            if match:
                title = match.group(1).strip()
                # Normalize smart/curly quotes to straight ASCII for matching
                title_normalized = title.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
                current_scenario_title = title
                current_scenario_key = SCENARIO_TITLE_TO_KEY.get(title_normalized)
                if current_scenario_key is None:
                    # Also try the original title in case the dict uses smart quotes
                    current_scenario_key = SCENARIO_TITLE_TO_KEY.get(title)
                if current_scenario_key is None:
                    log.warning("Unknown scenario title: %r", title)
            current_subsection = None
            continue

        if text.startswith("Target traits:"):
            current_target_traits = text.replace("Target traits:", "").strip()
            continue

        if style == "Heading 3":
            if text.startswith("Roleplay LLM"):
                current_subsection = "roleplayer"
            elif text.startswith("Judge LLM"):
                current_subsection = "antagonist"
            else:
                current_subsection = None
            continue

        if current_subsection == "roleplayer":
            roleplayer_buf.append(text)
        elif current_subsection == "antagonist":
            antagonist_buf.append(text)

    flush_cell()
    log.info("Extracted %d cells from docx", len(cells))
    return cells


# ===========================================================================
# PROVIDER CALL WRAPPERS
# ===========================================================================
# Each wrapper is a pure function: (system, messages, model, max_tokens) -> text.
# A fresh client is created per call. No shared state between calls.
#
# "messages" uses the universal form: [{"role": "user"|"assistant", "content": str}, ...]
# System prompt is passed separately.

async def call_zhipuai(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    """ZhipuAI (CharacterGLM) call. Uses OpenAI-compatible interface."""
    try:
        from zhipuai import ZhipuAI
    except ImportError:
        log.error("zhipuai is required. Install with: pip install zhipuai")
        sys.exit(1)

    client = ZhipuAI(api_key=os.environ.get("ZHIPUAI_API_KEY"))
    # CharacterGLM expects system message in the messages list (OpenAI-style)
    full_messages = [{"role": "system", "content": system}] + messages

    # ZhipuAI's SDK is synchronous; run it in a thread to keep the async loop unblocked.
    def _call():
        response = client.chat.completions.create(
            model=model,
            messages=full_messages,
            max_tokens=max_tokens,
            temperature=0.95,
        )
        return response.choices[0].message.content or ""

    text = await asyncio.to_thread(_call)
    return text.strip()


async def call_anthropic(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    """Anthropic (Claude) call."""
    try:
        from anthropic import AsyncAnthropic
    except ImportError:
        log.error("anthropic is required. Install with: pip install anthropic")
        sys.exit(1)

    client = AsyncAnthropic()  # reads ANTHROPIC_API_KEY
    response = await client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=1.0,
        system=system,
        messages=messages,
    )
    parts = [b.text for b in response.content if getattr(b, "type", None) == "text"]
    return "".join(parts).strip()


async def call_openai(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    """OpenAI (GPT-4o) call."""
    try:
        from openai import AsyncOpenAI
    except ImportError:
        log.error("openai is required. Install with: pip install openai")
        sys.exit(1)

    client = AsyncOpenAI()  # reads OPENAI_API_KEY
    full_messages = [{"role": "system", "content": system}] + messages
    response = await client.chat.completions.create(
        model=model,
        messages=full_messages,
        max_tokens=max_tokens,
        temperature=1.0,
    )
    return (response.choices[0].message.content or "").strip()

try:
    from google import genai
except ImportError:
    log.error("google-generativeai is required. Install with: pip install google-generativeai")
    sys.exit(1)

async def call_gemini(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))
    contents = []
    for m in messages:
        role = "model" if m["role"] == "assistant" else "user"
        contents.append({"role": role, "parts": [{"text": m["content"]}]})

    def _call():
        response = client.models.generate_content(
            model=model,
            contents=contents,
            config={"system_instruction": system, "max_output_tokens": max_tokens, "temperature": 1.0},
        )
        return response.text or ""

    text = await asyncio.to_thread(_call)
    return text.strip()


async def call_deepseek(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    """DeepSeek call. Uses OpenAI-compatible endpoint."""
    try:
        from openai import AsyncOpenAI
    except ImportError:
        log.error("openai is required for DeepSeek. Install with: pip install openai")
        sys.exit(1)

    client = AsyncOpenAI(
        api_key=os.environ.get("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com",
    )
    full_messages = [{"role": "system", "content": system}] + messages
    response = await client.chat.completions.create(
        model=model,
        messages=full_messages,
        max_tokens=max_tokens,
        temperature=1.0,
    )
    return (response.choices[0].message.content or "").strip()


PROVIDER_DISPATCH = {
    "zhipuai":   call_zhipuai,
    "anthropic": call_anthropic,
    "openai":    call_openai,
    "gemini":    call_gemini,
    "deepseek":  call_deepseek,
}


# ---------------------------------------------------------------------------
# ChatHaruhi roleplayer wrapper
# ---------------------------------------------------------------------------
# ChatHaruhi's RAG is BYPASSED: we inject the docx system prompt directly
# into the underlying LLM call, making it comparable to CharacterGLM.
# The backend LLM defaults to OpenAI (gpt-4o) but can be overridden via
# CHATHARUHI_BACKEND env var (openai / claude / local).

async def call_chatharuhi(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    """ChatHaruhi call — injects system prompt into backend LLM, bypassing RAG.

    We use the ChatHaruhi library's generate_prompt() to get the prompt format,
    but override persona with our docx system prompt. If ChatHaruhi is not installed,
    we fall back to calling the backend LLM directly with the system prompt
    (which is functionally identical when RAG is bypassed).
    """
    try:
        from chatharuhi import ChatHaruhi as _ChatHaruhi
        _HAS_CHATHARUHI = True
    except ImportError:
        _HAS_CHATHARUHI = False

    backend = os.environ.get("CHATHARUHI_BACKEND", "openai").lower()

    if _HAS_CHATHARUHI:
        # Use ChatHaruhi with custom persona (no RAG DB = no retrieval)
        def _call():
            chatbot = _ChatHaruhi(
                system_prompt=system,
                llm=backend,
                story_text_folder=None,  # no RAG
            )
            # Feed conversation history
            for msg in messages[:-1]:
                if msg["role"] == "user":
                    chatbot.chat(role="user", text=msg["content"])
            # Generate response for the last user message
            latest = messages[-1]["content"] if messages else ""
            response = chatbot.chat(role="user", text=latest)
            return response
        text = await asyncio.to_thread(_call)
        # ChatHaruhi often prefixes "CharacterName:「" — strip that
        text = re.sub(r"^[^:：]+[:：]\s*[「「]?", "", text.strip())
        text = text.rstrip("」」")
        return text.strip()
    else:
        # Fallback: call backend LLM directly with the system prompt
        log.warning("chatharuhi not installed; falling back to direct %s call with system prompt", backend)
        if backend == "openai":
            return await call_openai(system, messages, "gpt-4o", max_tokens)
        elif backend == "claude":
            return await call_anthropic(system, messages, "claude-sonnet-4-5", max_tokens)
        else:
            raise ValueError(f"Unsupported CHATHARUHI_BACKEND: {backend}")


# ---------------------------------------------------------------------------
# RoleLLM roleplayer wrapper (vLLM OpenAI-compatible endpoint)
# ---------------------------------------------------------------------------
# zephyr7788/RoleLLM (Qwen1.5-7B-Chat based) served via vLLM exposes an OpenAI-compatible /v1/chat/completions.
# The docx system prompt is injected directly. This uses the model off its
# trained instruction format — note this in writeup as a known limitation.

ROLELLM_INSTRUCTION_PREFIX = (
    "You are now in roleplay conversation mode. Pretend to be the character "
    "described below. Stay in-character at all times and generate responses "
    "as that character would.\n\n"
)

async def call_rolellm(system: str, messages: list[dict[str, str]], model: str, max_tokens: int) -> str:
    """RoleLLM call via local vLLM OpenAI-compatible endpoint."""
    try:
        from openai import AsyncOpenAI
    except ImportError:
        log.error("openai is required for RoleLLM. Install with: pip install openai")
        sys.exit(1)

    base_url = os.environ.get("ROLELLM_API_BASE", "http://localhost:8000/v1")
    client = AsyncOpenAI(
        api_key="EMPTY",  # vLLM doesn't need a real key
        base_url=base_url,
    )
    # Prepend RoleLLM-style instruction framing before the docx system prompt
    full_system = ROLELLM_INSTRUCTION_PREFIX + system
    full_messages = [{"role": "system", "content": full_system}] + messages

    # Auto-detect model name from vLLM if not specified
    vllm_model = os.environ.get("ROLELLM_MODEL_NAME", model)

    response = await client.chat.completions.create(
        model=vllm_model,
        messages=full_messages,
        max_tokens=max_tokens,
        temperature=0.95,
    )
    return (response.choices[0].message.content or "").strip()


# ---------------------------------------------------------------------------
# Roleplayer dispatcher (swappable via --roleplayer CLI)
# ---------------------------------------------------------------------------

async def call_roleplayer(system: str, messages: list[dict[str, str]], roleplayer_key: str) -> str:
    """Dispatch to the selected roleplayer."""
    cfg = ROLEPLAYER_MODELS[roleplayer_key]
    provider = cfg["provider"]
    model = cfg["model"]
    if provider == "zhipuai":
        return await call_zhipuai(system, messages, model, ROLEPLAYER_MAX_TOKENS)
    elif provider == "chatharuhi":
        return await call_chatharuhi(system, messages, model, ROLEPLAYER_MAX_TOKENS)
    elif provider == "rolellm":
        return await call_rolellm(system, messages, model, ROLEPLAYER_MAX_TOKENS)
    else:
        raise ValueError(f"Unknown roleplayer provider: {provider}")


async def call_judge(
    system: str,
    messages: list[dict[str, str]],
    judge_key: str,
    *,
    max_tokens: int,
) -> str:
    """Dispatch to the selected judge provider."""
    cfg = JUDGE_MODELS[judge_key]
    provider = cfg["provider"]
    model = cfg["model"]
    fn = PROVIDER_DISPATCH[provider]
    return await fn(system, messages, model, max_tokens)


# ===========================================================================
# CONVERSATION RUNNER
# ===========================================================================

async def run_conversation(
    roleplayer_system: str,
    antagonist_system: str,
    turns: int,
    judge_key: str,
    roleplayer_key: str,
    session_id: str,
) -> list[dict[str, str]]:
    """
    Drive one two-sided conversation for `turns` total utterances.

    SESSION ISOLATION GUARANTEE:
    Both message histories are created EMPTY at the start of this function
    call. Nothing is inherited from previous trials, scenarios, or characters.

    Returns a transcript: list of {"speaker": "roleplayer"|"antagonist", "text": ...}
    """
    # HARD RESET: empty lists every call.
    antagonist_history: list[dict[str, str]] = []
    roleplayer_history: list[dict[str, str]] = []
    assert len(antagonist_history) == 0 and len(roleplayer_history) == 0, \
        "Session isolation violated: histories must start empty"

    transcript: list[dict[str, str]] = []
    log.debug("session %s: starting fresh conversation (turns=%d)", session_id, turns)

    # Turn 1: antagonist opens. Seed with a minimal user prompt.
    antagonist_history.append({
        "role": "user",
        "content": "Begin the scenario now. Speak first, in character, as instructed.",
    })
    opening = await call_judge(
        antagonist_system, antagonist_history, judge_key, max_tokens=ANTAGONIST_MAX_TOKENS
    )
    antagonist_history.append({"role": "assistant", "content": opening})
    transcript.append({"speaker": "antagonist", "text": opening})

    # Roleplayer hears the opening
    roleplayer_history.append({"role": "user", "content": opening})

    # Alternate until we hit the turn cap (enforced in code, not in prompt)
    while len(transcript) < turns:
        # Roleplayer (CharacterGLM) responds
        rp_reply = await call_roleplayer(roleplayer_system, roleplayer_history, roleplayer_key)
        roleplayer_history.append({"role": "assistant", "content": rp_reply})
        transcript.append({"speaker": "roleplayer", "text": rp_reply})
        if len(transcript) >= turns:
            break
        antagonist_history.append({"role": "user", "content": rp_reply})

        # Antagonist (judge model) responds
        ant_reply = await call_judge(
            antagonist_system, antagonist_history, judge_key, max_tokens=ANTAGONIST_MAX_TOKENS
        )
        antagonist_history.append({"role": "assistant", "content": ant_reply})
        transcript.append({"speaker": "antagonist", "text": ant_reply})
        if len(transcript) >= turns:
            break
        roleplayer_history.append({"role": "user", "content": ant_reply})

    return transcript


# ===========================================================================
# EVALUATOR (judge model in a FRESH session)
# ===========================================================================

EVALUATOR_SYSTEM = """You are an expert psychological evaluator trained in Big Five personality assessment.

You will be given a multi-turn dialogue transcript in which someone is roleplaying a fictional character. Your job is to score the character's Big Five personality traits based ONLY on their behavior and language in the transcript.

You are a NEUTRAL evaluator. You are not a participant in the scenario. You have no stake in the outcome of the conversation. Do not be swayed by whether the character was "rude" or "nice" to the other speaker — focus on what their behavior reveals about their underlying traits.

Scoring scale (1-5):
  1 = Very low on this trait
  2 = Low
  3 = Moderate / mixed
  4 = High
  5 = Very high

For each trait, cite one SPECIFIC short quote or behavior from the transcript as your evidence. Keep each behavioral_evidence field to 1-2 short sentences, maximum 40 words. Do not rely on your prior knowledge of the character — score only from the dialogue provided.

Output format: a single JSON object with exactly this structure:
{
  "Openness":          {"score": <1-5>, "behavioral_evidence": "<brief citation>"},
  "Conscientiousness": {"score": <1-5>, "behavioral_evidence": "<brief citation>"},
  "Extraversion":      {"score": <1-5>, "behavioral_evidence": "<brief citation>"},
  "Agreeableness":     {"score": <1-5>, "behavioral_evidence": "<brief citation>"},
  "Neuroticism":       {"score": <1-5>, "behavioral_evidence": "<brief citation>"}
}

Output ONLY the JSON object. No preamble, no markdown code fences, no trailing commentary.
"""


def format_transcript_for_evaluator(transcript: list[dict[str, str]], character: str) -> str:
    """Render the transcript in a speaker-labelled form for the evaluator."""
    lines = [f"Transcript (the roleplayer is playing {character}):", ""]
    for turn in transcript:
        label = character if turn["speaker"] == "roleplayer" else "OTHER"
        lines.append(f"{label}: {turn['text']}")
        lines.append("")
    return "\n".join(lines)


async def evaluate_transcript(
    transcript: list[dict[str, str]],
    character: str,
    judge_key: str,
    session_id: str,
) -> dict[str, Any]:
    """
    Score the transcript in a COMPLETELY FRESH session.

    Brand-new API call with a brand-new message list containing ONLY the
    evaluator system prompt and the formatted transcript. No conversation
    history leaks in from the roleplay session above.
    """
    formatted = format_transcript_for_evaluator(transcript, character)
    messages: list[dict[str, str]] = [{"role": "user", "content": formatted}]
    assert len(messages) == 1, "Evaluator session must start with exactly one user message"

    log.debug("session %s: evaluator call (judge=%s)", session_id, judge_key)
    raw = await call_judge(
        EVALUATOR_SYSTEM, messages, judge_key, max_tokens=EVALUATOR_MAX_TOKENS
    )

    # Parse JSON leniently: strip code fences, then fall back to first {...} block.
    parsed: dict[str, Any] | None = None
    parse_error: str | None = None

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
            except json.JSONDecodeError as e2:
                parse_error = f"{e2}"
        else:
            parse_error = f"{e}"

    return {"raw_response": raw, "parsed": parsed, "parse_error": parse_error}


# ===========================================================================
# TRIAL ORCHESTRATION
# ===========================================================================

@dataclass
class TrialResult:
    character: str
    scenario_key: str
    scenario_title: str
    trial_index: int
    session_id: str
    turns: int
    judge_key: str
    judge_model: str
    roleplayer_model: str
    transcript: list[dict[str, str]]
    evaluator_raw: str
    evaluator_parsed: dict[str, Any] | None
    evaluator_parse_error: str | None
    ground_truth: dict[str, int]
    ground_truth_confidence: dict[str, float]
    mae: float | None = None


def compute_mae(predicted: dict[str, Any] | None, truth: dict[str, int]) -> float | None:
    """Mean absolute error between parsed scores and ground truth."""
    if predicted is None:
        return None
    try:
        errors = []
        for trait in TRAIT_NAMES:
            pred_entry = predicted.get(trait)
            if not isinstance(pred_entry, dict) or "score" not in pred_entry:
                return None
            pred_score = float(pred_entry["score"])
            errors.append(abs(pred_score - truth[trait]))
        return sum(errors) / len(errors)
    except (TypeError, ValueError, KeyError):
        return None


def trial_output_path(out_dir: Path, character: str, scenario_key: str, trial_idx: int) -> Path:
    safe_char = re.sub(r"[^A-Za-z0-9_]+", "_", character)
    return out_dir / safe_char / f"{scenario_key}_trial{trial_idx}.json"


async def run_trial(
    cell: Cell,
    truth: GroundTruth,
    trial_idx: int,
    turns: int,
    judge_key: str,
    roleplayer_key: str,
    out_dir: Path,
    semaphore: asyncio.Semaphore,
) -> TrialResult | None:
    """
    Run one (character, scenario, trial) cell in COMPLETE ISOLATION.

    Isolation boundary: creates its own session_id, calls run_conversation
    which builds fresh histories from scratch, then calls evaluate_transcript
    with a brand-new message list. Nothing from any other trial is visible.
    """
    out_path = trial_output_path(out_dir, cell.character, cell.scenario_key, trial_idx)
    if out_path.exists():
        log.info("SKIP  %s / %s / trial %d (already done)", cell.character, cell.scenario_key, trial_idx)
        return None

    out_path.parent.mkdir(parents=True, exist_ok=True)
    session_id = uuid.uuid4().hex[:12]

    async with semaphore:
        log.info("START %s / %s / trial %d (session %s, judge=%s, roleplayer=%s, turns=%d)",
                 cell.character, cell.scenario_key, trial_idx, session_id, judge_key, roleplayer_key, turns)
        try:
            transcript = await run_conversation(
                roleplayer_system=cell.roleplayer_prompt,
                antagonist_system=cell.antagonist_prompt,
                turns=turns,
                judge_key=judge_key,
                roleplayer_key=roleplayer_key,
                session_id=session_id,
            )
            evaluation = await evaluate_transcript(
                transcript, cell.character, judge_key, session_id=session_id
            )
        except Exception as e:
            log.exception("FAIL  %s / %s / trial %d: %s",
                          cell.character, cell.scenario_key, trial_idx, e)
            return None

        roleplayer_model = ROLEPLAYER_MODELS[roleplayer_key]["model"]
        result = TrialResult(
            character=cell.character,
            scenario_key=cell.scenario_key,
            scenario_title=cell.scenario_title,
            trial_index=trial_idx,
            session_id=session_id,
            turns=turns,
            judge_key=judge_key,
            judge_model=JUDGE_MODELS[judge_key]["model"],
            roleplayer_model=roleplayer_model,
            transcript=transcript,
            evaluator_raw=evaluation["raw_response"],
            evaluator_parsed=evaluation["parsed"],
            evaluator_parse_error=evaluation["parse_error"],
            ground_truth=truth.as_dict(),
            ground_truth_confidence=truth.confidence,
        )
        result.mae = compute_mae(evaluation["parsed"], truth.as_dict())

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(asdict(result), f, ensure_ascii=False, indent=2)

        mae_str = f"MAE={result.mae:.2f}" if result.mae is not None else "MAE=parse_fail"
        log.info("DONE  %s / %s / trial %d  %s  session=%s",
                 cell.character, cell.scenario_key, trial_idx, mae_str, session_id)
        return result


# ===========================================================================
# SUMMARY CSV
# ===========================================================================

def write_summary_csv(out_dir: Path) -> None:
    """Flatten all trial JSON files into one CSV for analysis."""
    rows = []
    for json_path in sorted(out_dir.rglob("*_trial*.json")):
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        parsed = data.get("evaluator_parsed") or {}
        row = {
            "character": data["character"],
            "scenario_key": data["scenario_key"],
            "scenario_title": data["scenario_title"],
            "trial_index": data["trial_index"],
            "session_id": data.get("session_id", ""),
            "turns": data["turns"],
            "judge_key": data.get("judge_key", ""),
            "judge_model": data.get("judge_model", ""),
            "roleplayer_model": data.get("roleplayer_model", ""),
            "mae": data.get("mae"),
            "parse_ok": data.get("evaluator_parse_error") is None,
        }
        for trait in TRAIT_NAMES:
            truth = data["ground_truth"][trait]
            conf = data["ground_truth_confidence"][trait]
            pred_entry = parsed.get(trait) if isinstance(parsed, dict) else None
            pred_score = pred_entry.get("score") if isinstance(pred_entry, dict) else None
            row[f"truth_{trait}"] = truth
            row[f"pred_{trait}"] = pred_score
            row[f"conf_{trait}"] = conf
            row[f"abserr_{trait}"] = abs(pred_score - truth) if pred_score is not None else None
        rows.append(row)

    if not rows:
        log.warning("No trial files found; summary CSV not written.")
        return

    summary_path = out_dir / "summary.csv"
    with open(summary_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    log.info("Wrote summary with %d rows to %s", len(rows), summary_path)


# ===========================================================================
# ENV VAR CHECK
# ===========================================================================

def check_env_vars(judge_key: str, roleplayer_key: str) -> None:
    """Verify required API keys are set."""
    required = []

    # Roleplayer env vars
    roleplayer_env = {
        "charglm":    "ZHIPUAI_API_KEY",
        "chatharuhi": "OPENAI_API_KEY",   # default backend; override via CHATHARUHI_BACKEND
        "rolellm":    None,                # vLLM local, no key needed
    }
    rp_env = roleplayer_env.get(roleplayer_key)
    if rp_env:
        # ChatHaruhi backend might use Claude instead of OpenAI
        if roleplayer_key == "chatharuhi":
            backend = os.environ.get("CHATHARUHI_BACKEND", "openai").lower()
            if backend == "claude":
                rp_env = "ANTHROPIC_API_KEY"
        required.append(rp_env)

    # Judge env vars
    judge_env = {
        "claude":   "ANTHROPIC_API_KEY",
        "openai":   "OPENAI_API_KEY",
        "gemini":   "GOOGLE_API_KEY",
        "deepseek": "DEEPSEEK_API_KEY",
    }
    required.append(judge_env[judge_key])

    # Deduplicate (e.g. both judge and roleplayer need OPENAI_API_KEY)
    required = list(set(required))

    missing = [v for v in required if not os.environ.get(v)]
    if missing:
        log.error("Missing required environment variables: %s", ", ".join(missing))
        sys.exit(1)


# ===========================================================================
# MAIN
# ===========================================================================

async def main_async(args) -> None:
    check_env_vars(args.judge, args.roleplayer)

    truths = load_ground_truth(Path(args.csv))
    cells = extract_cells_from_docx(Path(args.docx))

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Sanity: characters in CSV vs docx
    csv_chars = set(truths.keys())
    docx_chars = {c for (c, _) in cells.keys()}
    missing_in_docx = csv_chars - docx_chars
    missing_in_csv = docx_chars - csv_chars
    if missing_in_docx:
        log.warning("Characters in CSV but not in docx: %s", missing_in_docx)
    if missing_in_csv:
        log.warning("Characters in docx but not in CSV: %s", missing_in_csv)

    roleplayer_model = ROLEPLAYER_MODELS[args.roleplayer]["model"]
    log.info("Roleplayer: %s (%s)", args.roleplayer, roleplayer_model)
    log.info("Judge: %s (%s)", args.judge, JUDGE_MODELS[args.judge]["model"])
    log.info("Turns per conversation: %d", args.turns)
    log.info("Trials per cell: %d", args.trials)
    log.info("Concurrency: %d", args.concurrency)

    # Build task list. Each trial is a fully isolated task.
    semaphore = asyncio.Semaphore(args.concurrency)
    tasks = []
    for (character, scenario_key), cell in sorted(cells.items()):
        if character not in truths:
            log.warning("Skipping %s / %s: no ground truth", character, scenario_key)
            continue
        truth = truths[character]
        for trial_idx in range(args.trials):
            tasks.append(run_trial(
                cell=cell,
                truth=truth,
                trial_idx=trial_idx,
                turns=args.turns,
                judge_key=args.judge,
                roleplayer_key=args.roleplayer,
                out_dir=out_dir,
                semaphore=semaphore,
            ))

    log.info("Running %d isolated trials", len(tasks))
    await asyncio.gather(*tasks)
    write_summary_csv(out_dir)
    log.info("All done.")



# ===========================================================================
# NOTEBOOK ENTRY POINT
# ===========================================================================

async def run_experiment(
    roleplayer: str,
    judge: str,
    csv_path: str = "characterglm_roles.csv",
    docx_path: str = "prompt_template_customized_new.docx",
    out: str = "results",
    trials: int = 3,
    turns: int = 10,
    concurrency: int = 4,
) -> None:
    """Run one (roleplayer, judge) experiment combo from a notebook cell."""
    import types
    args = types.SimpleNamespace(
        roleplayer=roleplayer,
        judge=judge,
        csv=csv_path,
        docx=docx_path,
        out=out,
        trials=trials,
        turns=turns,
        concurrency=concurrency,
    )
    if args.turns < 2:
        raise ValueError("turns must be >= 2")
    await main_async(args)

print("✅ Pipeline loaded. Use run_experiment() in the next cell.")


✅ Pipeline loaded. Use run_experiment() in the next cell.


## Cell 5 — (Optional) Launch vLLM for RoleLLaMA

**Only run this if you plan to use `roleplayer='rolellm'`.**

Make sure you selected **A100 GPU** runtime first. Takes ~3-5 min to download and start.

In [ ]:
# Install vLLM
!pip install -q vllm

import subprocess, time, requests, os

vllm_proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "zephyr7788/RoleLLM",
     "--dtype", "float16",
     "--max-model-len", "8192",
     "--port", "8000",
     "--gpu-memory-utilization", "0.85"],
    stdout=open("/content/vllm_stdout.log", "w"),
    stderr=open("/content/vllm_stderr.log", "w"),
)

print("Waiting for vLLM to start...")
for i in range(120):
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=2)
        if r.status_code == 200:
            models = r.json()
            model_name = models["data"][0]["id"]
            os.environ["ROLELLM_MODEL_NAME"] = model_name
            print(f"✅ vLLM ready! Model: {model_name}")
            break
    except: pass
    time.sleep(2)
    if i % 10 == 9: print(f"   Still waiting... ({(i+1)*2}s)")
else:
    print("❌ vLLM did not start. Check /content/vllm_stderr.log")
    !tail -20 /content/vllm_stderr.log


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
langchain-core 0.1.53 requires packaging<24.0,>=23.2, but you have packaging 26.1 which is incompatible.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
langgraph-checkpoint 4.0.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.2 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which i

## Cell 6 — Run Experiments

Uncomment the combos you want. Each line runs one `(roleplayer, judge)` pair.

**Cost estimate per combo** (3 chars × 6 scenarios × 3 trials × 10 turns):
- CharacterGLM + any judge ≈ $2-4
- ChatHaruhi + any judge ≈ $3-5 (GPT-4o backbone + judge)
- RoleLLaMA + any judge ≈ $1-2 (local model, only judge API costs)

In [ ]:
! rm -r results/chatharuhi_gemini

In [ ]:
!pip install -q "openai>=1.40.0" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
!curl -s http://localhost:8000/v1/models || echo "vLLM is NOT running"

vLLM is NOT running


In [ ]:
!pip install numpy>=2.0 scipy --upgrade --break-system-packages
!pip install scipy==1.13.1 --break-system-packages

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.4.4 which is incompatible.
flashinfer-python 0.6.6 requires packaging>=24.2, but you have packaging 23.2 which is incompatible.
mistral-common 1.11.0 requires numpy<2.4,>=1.25; python_version <= "3.12", but you have numpy 2.4.4 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.4 which is incompatible.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
db-dtypes 1.5.1 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.2 which is incompatible.
xarray 2025.12.0 requires packaging>=2

In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()  # needed for asyncio.run() inside Colab/Jupyter

# ═══════════════════════════════════════════════════════════════
# Uncomment the combos you want to run
# ═══════════════════════════════════════════════════════════════

# --- CharacterGLM (requires ZHIPUAI_API_KEY) ---
# await run_experiment(roleplayer="charglm", judge="claude",   out="results/charglm_claude")
# await run_experiment(roleplayer="charglm", judge="openai",   out="results/charglm_openai")
# await run_experiment(roleplayer="charglm", judge="gemini",   out="results/charglm_gemini")
# await run_experiment(roleplayer="charglm", judge="deepseek", out="results/charglm_deepseek")

# --- ChatHaruhi (requires OPENAI_API_KEY for backbone + judge key) ---
#await run_experiment(roleplayer="chatharuhi", judge="claude",   out="results/chatharuhi_claude", turns=20, trials=5, concurrency=2)
#await run_experiment(roleplayer="chatharuhi", judge="openai",   out="results/chatharuhi_openai", turns=20, trials=5, concurrency=1)
await run_experiment(roleplayer="chatharuhi", judge="gemini",   out="results/chatharuhi_gemini", turns=20, trials=5, concurrency=2)
#await run_experiment(roleplayer="chatharuhi", judge="deepseek", out="results/chatharuhi_deepseek", turns=20, trials=5, concurrency=2)

# --- RoleLLaMA (requires Cell 5 vLLM running + judge key) ---
#await run_experiment(roleplayer="rolellm", judge="claude",   out="results/rolellm_claude", turns=20, trials=5, concurrency=2)
#await run_experiment(roleplayer="rolellm", judge="openai",   out="results/rolellm_openai", turns=20, trials=5, concurrency=1)
# await run_experiment(roleplayer="rolellm", judge="gemini",   out="results/rolellm_gemini", turns=20, trials=5, concurrency=1)
#await run_experiment(roleplayer="rolellm", judge="deepseek", out="results/rolellm_deepseek", turns=20, trials=5, concurrency=2)


warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_folder are not inputted.
warning! database not yet figured out, both story_db and story_text_fo

## Cell 7 — Check Results

In [ ]:
import os, json, glob, csv as csv_mod

print("=== Results Summary ===")
for result_dir in sorted(glob.glob("results/*/")):
    jsons = glob.glob(os.path.join(result_dir, "**/*_trial*.json"), recursive=True)
    csvs = glob.glob(os.path.join(result_dir, "summary.csv"))
    print(f"  {result_dir}: {len(jsons)} trials, summary={'YES' if csvs else 'NO'}")
    if csvs:
        with open(csvs[0]) as f:
            rows = list(csv_mod.DictReader(f))
        maes = [float(r["mae"]) for r in rows if r.get("mae") and r["mae"] != "None"]
        if maes:
            print(f"    MAE: mean={sum(maes)/len(maes):.2f}, "
                  f"min={min(maes):.2f}, max={max(maes):.2f}, "
                  f"n={len(maes)}/{len(rows)} parsed")


=== Results Summary ===
  results/chatharuhi_claude/: 90 trials, summary=YES
    MAE: mean=1.08, min=0.20, max=2.00, n=90/90 parsed
  results/chatharuhi_deepseek/: 90 trials, summary=YES
    MAE: mean=1.06, min=0.00, max=2.00, n=90/90 parsed
  results/chatharuhi_openai/: 90 trials, summary=YES
    MAE: mean=1.12, min=0.20, max=1.80, n=90/90 parsed
  results/rolellm_claude/: 90 trials, summary=YES
    MAE: mean=1.35, min=0.40, max=3.20, n=90/90 parsed
  results/rolellm_deepseek/: 90 trials, summary=YES
    MAE: mean=1.20, min=0.20, max=2.80, n=90/90 parsed
  results/rolellm_openai/: 90 trials, summary=YES
    MAE: mean=1.14, min=0.20, max=2.00, n=90/90 parsed


## Cell 8 — Download Results

In [ ]:
!zip -r /content/results_all.zip results/
from google.colab import files
files.download("/content/results_all.zip")
print("✅ Download started.")


updating: results/ (stored 0%)
updating: results/chatharuhi_claude/ (stored 0%)
updating: results/chatharuhi_claude/Thor/ (stored 0%)
updating: results/chatharuhi_claude/Thor/peer_confrontation_trial3.json (deflated 63%)
updating: results/chatharuhi_claude/Thor/high_stakes_evaluation_trial1.json (deflated 66%)
updating: results/chatharuhi_claude/Thor/defective_purchase_trial1.json (deflated 66%)
updating: results/chatharuhi_claude/Thor/defective_purchase_trial0.json (deflated 65%)
updating: results/chatharuhi_claude/Thor/authority_confrontation_trial2.json (deflated 66%)
updating: results/chatharuhi_claude/Thor/grave_news_trial4.json (deflated 67%)
updating: results/chatharuhi_claude/Thor/defective_purchase_trial2.json (deflated 64%)
updating: results/chatharuhi_claude/Thor/peer_confrontation_trial2.json (deflated 64%)
updating: results/chatharuhi_claude/Thor/peer_confrontation_trial0.json (deflated 63%)
updating: results/chatharuhi_claude/Thor/apprentice_confusion_trial3.json (deflate

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started.


## Cell 9 — (Optional) Stop vLLM Server

In [ ]:
!pkill -f "vllm.entrypoints" || echo "No vLLM process found"
print("vLLM stopped.")


In [ ]:
# Kill all background processes and end session
!pkill -f "vllm.entrypoints" 2>/dev/null
from google.colab import runtime
runtime.unassign()

^C


RuntimeManagementError: Unable to request VM unassignment.

In [ ]:
!unzip results.zip

Archive:  results.zip
   creating: results/
   creating: results/chatharuhi_openai/
  inflating: results/chatharuhi_openai/summary.csv  
   creating: results/chatharuhi_openai/Abraham_Lincoln/
  inflating: results/chatharuhi_openai/Abraham_Lincoln/apprentice_confusion_trial0.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/authority_confrontation_trial1.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/peer_confrontation_trial4.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/apprentice_confusion_trial1.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/high_stakes_evaluation_trial3.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/apprentice_confusion_trial2.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/defective_purchase_trial3.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/peer_confrontation_trial1.json  
  inflating: results/chatharuhi_openai/Abraham_Lincoln/peer_confrontation_trial3.json  
